In [ ]:
import cv2
import mediapipe as mp
import time
from datetime import datetime
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub

# Mediapipe setup
mp_face_mesh = mp.solutions.face_mesh
mp_pose = mp.solutions.pose

face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
pose = mp_pose.Pose(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

# SSD MobileNet model

ssd_model = hub.load("https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2")

coco_classes = {
    1: "person", 44: "bottle", 45: "wine glass", 46: "cup",
    47: "fork", 48: "knife", 49: "spoon",
    77: "cell phone", 85: "toothbrush", 88: "gun"
}
relevant_classes = ["knife", "gun"]

# SSD helpers (UNCHANGED)

def preprocess_frame(frame):
    resized_frame = tf.image.resize(frame, (300, 300))
    resized_frame = tf.cast(resized_frame, tf.uint8)
    return tf.expand_dims(resized_frame, axis=0)

def draw_boxes(frame, detections, scores, classes, threshold=0.25):
    h, w, _ = frame.shape
    for i in range(len(scores)):
        if scores[i] > threshold:
            class_id = int(classes[i])
            if class_id in coco_classes and coco_classes[class_id] in relevant_classes:
                ymin, xmin, ymax, xmax = detections[i]
                x1, y1 = int(xmin * w), int(ymin * h)
                x2, y2 = int(xmax * w), int(ymax * h)

                cv2.rectangle(frame, (x1, y1), (x2, y2),
                              (0, 0, 255), 2)

                label = coco_classes[class_id]
                cv2.putText(frame, f"{label} {scores[i]:.2f}",
                            (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                            (0, 0, 255), 2)

# 🔴 REQUIRED ADDITION (Method 1)

def landmark_visible(lm, edges, w, h):
    x = int(lm.x * w)
    y = int(lm.y * h)

    r = 6
    patch = edges[max(0, y-r):min(h, y+r),
                  max(0, x-r):min(w, x+r)]

    if patch.size == 0:
        return False

    return np.mean(patch) > 3


def calculate_face_coverage(face_landmarks, edges, w, h):
    critical_landmarks = [33, 133, 362, 263, 1, 2, 98, 324, 13, 14]
    visible = 0

    for idx in critical_landmarks:
        if landmark_visible(face_landmarks[idx], edges, w, h):
            visible += 1

    return (visible / len(critical_landmarks)) * 100

# Pose behavior (UNCHANGED)

def analyze_behavior(pose_landmarks):
    left_hand_y = pose_landmarks[mp_pose.PoseLandmark.LEFT_WRIST].y
    right_hand_y = pose_landmarks[mp_pose.PoseLandmark.RIGHT_WRIST].y
    nose_y = pose_landmarks[mp_pose.PoseLandmark.NOSE].y

    return left_hand_y < nose_y or right_hand_y < nose_y

# Webcam

cap = cv2.VideoCapture(0)

alert_message = "Warning: Potential Threat!"
alert_duration = 3
last_alert_time = 0

# Main loop

while True:
    success, frame = cap.read()
    if not success:
        break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray_frame, 80, 160)

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_results = face_mesh.process(image_rgb)
    pose_results = pose.process(image_rgb)

    h, w = frame.shape[:2]

    # Face coverage (NOW DYNAMIC)
    
    if face_results.multi_face_landmarks:
        for face_landmarks in face_results.multi_face_landmarks:
            coverage = calculate_face_coverage(
                face_landmarks.landmark, edges, w, h
            )

            print(f"Coverage: {coverage:.2f}% | Brightness: {np.mean(gray_frame):.2f}")


            if coverage < 20:
                current_time = time.time()
                if current_time - last_alert_time > alert_duration:
                    cv2.putText(frame, alert_message, (50, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1,
                                (0, 0, 255), 2)

                    filename = f"warning_image_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
                    cv2.imwrite(filename, frame)
                    last_alert_time = current_time

    # Unusual behavior (UNCHANGED)
    
    if pose_results.pose_landmarks:
        if analyze_behavior(pose_results.pose_landmarks.landmark):
            cv2.putText(frame, "Unusual Behavior Detected!",
                        (50, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 1,
                        (255, 0, 0), 2)

            filename = f"behavior_warning_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
            cv2.imwrite(filename, frame)

    # SSD object detection (UNCHANGED)
    
    preprocessed_frame = preprocess_frame(frame)
    result = ssd_model(preprocessed_frame)

    detections = result["detection_boxes"].numpy()[0]
    scores = result["detection_scores"].numpy()[0]
    classes = result["detection_classes"].numpy()[0]

    draw_boxes(frame, detections, scores, classes, threshold=0.5)

    cv2.imshow("Surveillance System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Coverage: 10.00% | Brightness: 154.15
Coverage: 10.00% | Brightness: 146.70
Coverage: 10.00% | Brightness: 146.73
Coverage: 10.00% | Brightness: 146.68
Coverage: 10.00% | Brightness: 146.66
Coverage: 10.00% | Brightness: 146.52
Coverage: 10.00% | Brightness: 147.58
Coverage: 10.00% | Brightness: 152.98
Coverage: 10.00% | Brightness: 156.87
Coverage: 0.00% | Brightness: 156.89
Coverage: 0.00% | Brightness: 157.07
Coverage: 0.00% | Brightness: 157.09
Coverage: 0.00% | Brightness: 157.17
Coverage: 0.00% | Brightness: 157.30
Coverage: 0.00% | Brightness: 157.39
Coverage: 0.00% | Brightness: 157.39
Coverage: 30.00% | Brightness: 157.42
Coverage: 30.00% | Brightness: 157.33
Coverage: 30.00% | Brightness: 157.05
Coverage: 30.00% | Brightness: 157.07
Coverage: 0.00% | Brightness: 157.05
Coverage: 0.00% | Brightness: 156.53
Coverage: 0.00% | Brightness: 156.58
Coverage: 0.00% | Brightness: 157.18
Coverage: 0.00% | Brightness: 157.32
Coverage: 30.00% | Brightness: 157.38
Coverage: 0.00% | Bright